# Your first project in Python

The U-bracket from its STL to the optimized design, in Python only: the same steps as the GUI's
sidebar, written as calls. Run the cells in order, from the top. Every call used here is described,
with its arguments and an example, in `docs/reference.md`, and in the GUI under
Help > Researcher guide > Reference; each note below links to the calls its cell uses.

## 1. The part and its units

One import line brings in everything the setup needs. [Project](docs/reference.md#project) makes a project
from the part's STL; `single_layer=True` builds one element through the thickness, since the
U-bracket is a flat plate. [p.units](docs/reference.md#punits) says which units the numbers are in: here
metres, newtons and kilograms (MKS). Nothing is converted; the units are checked against the
material database.

In [ ]:
from lsr_mmto import Project, run, plane, box, cylinder, union

p = Project("U-bracket", stl="data/models/UBracket/UBracket.STL", single_layer=True)
p.units = "MKS"

## 2. The material database

[p.database](docs/reference.md#pdatabase) names one of the shipped databases. It comes with a trained
network, which turns each material into a point of the material space the optimization moves in.
[p.network_error](docs/reference.md#pnetwork_error) gives the line the Materials panel shows: how closely the
network gives back each material. [p.vae_check](docs/reference.md#pvae_check) lists what stops a run from
using the network; it prints nothing when the network fits the settings. If it says the latent
dimension does not match, set `p.settings["vae"]["latentDim"]` to the number it gives, or set
`p.settings["vae"]["use_pretrained_vae"] = False` so that the run trains a new network.

In [ ]:
p.database = "6Materials_AF_3Attr"

rows, line = p.network_error()
print(line)
errors, warnings, status = p.vae_check()
print(status)
for problem in errors:
    print(problem)

## 3. Selections

A selection names a region of the part; supports, loads and keep solid then act on it by name.
[p.add_selection](docs/reference.md#padd_selection) adds one. [p.mesh](docs/reference.md#pmesh) builds the mesh the
run will use, and [p.evaluate](docs/reference.md#pevaluate) gives the nodes a selection picks on it. The small
function `show` below prints how many nodes a selection picks and draws them in red on the part.

In [ ]:
import matplotlib.pyplot as plt

mesh = p.mesh()
xyz = mesh.node_xyz


def show(name):
    picked = p.evaluate(name, mesh)
    print(f"{name}: {len(picked)} nodes")
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.scatter(xyz[:, 0], xyz[:, 1], s=1, color="lightgrey")
    ax.scatter(xyz[picked, 0], xyz[picked, 1], s=4, color="crimson")
    ax.set_title(name)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()

A [plane](docs/reference.md#plane) across x at x = 0: the left edge, where the part will be held.

In [ ]:
p.add_selection("left", plane("x", 0.0))
show("left")

A [box](docs/reference.md#box) at the top right corner, from one corner to the opposite one, in metres:
where the force will act.

In [ ]:
p.add_selection("tip", box([0.95, 0.9, 0], [1, 1, 0.01]))
show("tip")

A [cylinder](docs/reference.md#cylinder) through the plate (along z), 5 cm in radius: a region kept solid.

In [ ]:
p.add_selection("boss", cylinder(center=[0.875, 0.25, 0], axis="z", radius=0.05))
show("boss")

A combination: the [union](docs/reference.md#union) of two planes, each limited to one arm, gives the top
edges of both arms.

In [ ]:
p.add_selection("top", union(plane("y", 1.0, limits={"x": [0, 0.5]}), plane("y", 1.0, limits={"x": [0.75, 1]})))
show("top")

## 4. Supports and loads

[p.add_bc](docs/reference.md#padd_bc) puts a support or a load on a selection: the left edge held in x, y
and z, a total force of 1 N to the left at the tip, and a pressure of 10 Pa pushing down on the top
edges. The values each kind takes are listed under Boundary conditions in the reference.
[p.keep_solid](docs/reference.md#pkeep_solid) keeps the cylinder's elements solid in every design.

In [ ]:
p.add_bc("fixed", "left", dofs="xyz")
p.add_bc("force", "tip", total=[-1.0, 0.0, 0.0])
p.add_bc("traction", "top", pressure=10.0, direction=[0, -1, 0], area="bbox")
p.keep_solid("boss")

## 5. Objective and constraints

[p.objective](docs/reference.md#pobjective) is what the optimization makes as small as it can, here the
failure factor (the stress compared with each material's yield strength).
[p.constraint](docs/reference.md#pconstraint) adds a limit: the mass at most 3.5 g. Every quantity is listed
under Quantities in the reference.

In [ ]:
p.objective("FailureFactorPNorm()")
p.constraint("Mass()", "<=", 0.0035)

## 6. Settings

[p.settings](docs/reference.md#psettings) holds every setting of a run, by section. Below, the mesh size
(`n_dof`, see [Project](docs/reference.md#project)) and the number of iterations are set small, so that the
run finishes in a minute or two. A real design needs the defaults or more.

In [ ]:
print("n_dof", p.n_dof, " single_layer", p.single_layer)
for section, values in p.settings.items():
    print(section, values)

p.n_dof = 6000
p.settings["common"]["maxIterations"] = 30

## 7. Saving the project

[p.save](docs/reference.md#psave) writes `U-bracket.mmto` into the folder `python_walkthrough_project`, next
to this notebook. That folder becomes the project's folder: the part is copied into `part/` and the
database with its network into `materials/`, so the folder can be zipped and sent. The GUI opens the
same file (File > Open), with every step set as it is here.

In [ ]:
p.save("python_walkthrough_project/U-bracket.mmto")

## 8. The run

[run](docs/reference.md#run) optimizes the project and prints each iteration. Its results go into
`results`, in a folder named after the date and time, inside the project's folder; `run` returns
that folder. The GUI's Results lists the run as well.

In [ ]:
folder = run(p)
print(folder)

## 9. The results

[load](docs/reference.md#load) reads the run back from its folder ([Run fields](docs/reference.md#run-fields) lists
what it holds). [summary](docs/reference.md#summary) shows the result, the design, the materials and the run,
as the GUI's Summary does.

In [ ]:
from lsr_mmto.results.load_results import load
from lsr_mmto.results.tables import summary

r = load(folder)
summary(r)

[design](docs/reference.md#design) draws the design, each element in its material's colour.

In [ ]:
from lsr_mmto.results.run_plots import design

design(r)

[export](docs/reference.md#export) writes the design as smooth STL files into the run's folder, as Export STL
in the GUI does: the whole part, and one file for each material. It needs the mesh the run was solved
on, from [mesh_for](docs/reference.md#mesh_for). The lines say what was written and whether each file is closed.

In [ ]:
from lsr_mmto.results.design_3d import mesh_for
from lsr_mmto.results.stl_export import export

report = export(r, mesh_for(r), folder)
print("\n".join(report.lines("m")))